El conjunto de datos original Ames Housing contiene 82 columnas y 2930 registros que detallan absolutamente todas las características físicas y de zonificación de diversas casas (como el área del lote, el tipo de calle, la calidad de la cocina, el año de construcción, etc.), con el objetivo final de predecir tu variable objetivo: la columna SalePrice (el precio de venta de la vivienda).
En el modelo elimina automáticamente todas las columnas que contienen texto (como el tipo de calle o la zona) usando el comando select_dtypes(include=[np.number]). Se quitan todas estas características de texto porque la red neuronal matemática que estás construyendo desde cero con PyTorch en las siguientes celdas no sabe leer palabras, solo procesa números. Posteriormente, el código rellena cualquier hueco vacío con la mediana, y hace algo fundamental: separa y quita la columna SalePrice de las características de entrada (X) para evitar que la IA haga trampa leyendo la respuesta. Por último, a ese precio de venta (Y) se le aplica una fórmula de logaritmo (np.log1p) y se normaliza; esto se hace porque los precios de las casas varían muchísimo (unas cuestan $50,000 y otras $500,000), y al comprimir esos números gigantes, se ayuda a que el modelo no se desestabilice ni se vuelva loco durante el entrenamiento.

In [5]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("prevek18/ames-housing")

print("Path to dataset files:", path)

Path to dataset files: /Users/mac/.cache/kagglehub/datasets/prevek18/ames-housing/versions/2


In [6]:
# FASE 1: IMPORTS Y LIBRERÍAS
import os
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import kagglehub

# Herramientas básicas de PyTorch y Sklearn
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error

print("Fase 1 completada: Entorno listo.")

Fase 1 completada: Entorno listo.


In [9]:
# FASE 2: EXPLORACIÓN Y PREPARACIÓN DE DATOS
import os
import pandas as pd
import numpy as np
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# 1. Carga de datos desde tu ruta local
ruta_ames = '/Users/mac/Desktop/datasets/ Ames Housing Dataset/archive-4/AmesHousing.csv'

try:
    # Cargamos el archivo directamente
    df = pd.read_csv(ruta_ames)
    # Limpiamos los nombres de las columnas (quitamos espacios)
    df.columns = df.columns.str.replace(' ', '')
    
    print("Archivo cargado correctamente.")
    
    # ANÁLISIS EXPLORATORIO: REVISIÓN DE DATOS
    print("🔍 REVISIÓN DE LAS PRIMERAS 10 COLUMNAS:")
    display(df.iloc[:5, :10])

    print(f"\nTotal de registros: {df.shape[0]} | Total de columnas: {df.shape[1]}")
    print("-" * 60)

    # 2. Limpieza: Solo numéricas y rellenamos nulos
    # Seleccionamos solo columnas con números
    df_num = df.select_dtypes(include=[np.number])
    # Rellenamos los vacíos con la mediana de cada columna
    df_num = df_num.fillna(df_num.median())

    # 3. Separar X e Y (Y en logaritmo para normalizar precios altos)
    # Asegúrate de que 'SalePrice' sea el nombre exacto de la columna en tu CSV
    X_raw = df_num.drop('SalePrice', axis=1).values
    y_log = np.log1p(df_num['SalePrice'].values).reshape(-1, 1)

    # 4. Normalización (StandardScaler)
    scaler_X = StandardScaler()
    X_scaled = scaler_X.fit_transform(X_raw)

    scaler_Y = StandardScaler()
    y_scaled = scaler_Y.fit_transform(y_log)

    # 5. División de datos (80% entrenamiento, 20% prueba)
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_scaled, test_size=0.2, random_state=42)

    print(f" Fase 2 completada exitosamente.")
    print(f"Datos de entrenamiento: {X_train.shape[0]} | Datos de prueba: {X_test.shape[0]}")

except FileNotFoundError:
    print(f"ERROR: No se encontró el archivo en {ruta_ames}. Verifica que la carpeta 'archive-4' exista.")
except Exception as e:
    print(f" Ocurrió un error inesperado: {e}")

Archivo cargado correctamente.
🔍 REVISIÓN DE LAS PRIMERAS 10 COLUMNAS:


,Order,PID,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl



Total de registros: 2930 | Total de columnas: 82
------------------------------------------------------------
 Fase 2 completada exitosamente.
Datos de entrenamiento: 2344 | Datos de prueba: 586


In [10]:
# FASE 3: FUNCIÓN DE PÉRDIDA Y PESOS MANUALES (CPU)
import torch # <--- Línea indispensable
import numpy as np

# Definir el dispositivo
device = torch.device("cpu")

# Convertir datos de NumPy a Tensores de PyTorch
# Esto es necesario porque PyTorch no puede operar directamente con arreglos de NumPy
X_train_t = torch.from_numpy(X_train).float().to(device)
y_train_t = torch.from_numpy(y_train).float().to(device)

def mse_loss(output, target):
    loss = torch.mean((output - target)**2)
    return loss

# Arquitectura de la red
D_in, H, D_out = X_train.shape[1], 64, 1

# Inicialización de pesos (He initialization)
w1 = torch.tensor(np.random.normal(loc=0.0,
          scale = np.sqrt(2/(D_in+H)),
          size = (D_in, H)), requires_grad=True, device=device, dtype=torch.float)
b1 = torch.zeros(H, requires_grad=True, device=device, dtype=torch.float)

w2 = torch.tensor(np.random.normal(loc=0.0,
          scale = np.sqrt(2/(D_out+H)),
          size = (H, D_out)), requires_grad=True, device=device, dtype=torch.float)
b2 = torch.zeros(D_out, requires_grad=True, device=device, dtype=torch.float)

print("Fase 3 completada: Librería cargada, tensores creados y pesos inicializados.")

Fase 3 completada: Librería cargada, tensores creados y pesos inicializados.
